1. Import thư viện

In [ ]:
!pip install contractions spacy wordcloud torch transformers scikit-learn pandas numpy matplotlib seaborn nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# Xử lý ngôn ngữ tự nhiên (NLP)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
import spacy
from collections import Counter
# Các thư viện của Sklearn
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, auc, classification_report)


# Học sâu (Deep learning)
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from torch.utils.data import Dataset

# Wordcloud và NER
from wordcloud import WordCloud
nlp = spacy.load("en_core_web_sm")

# Tải dữ liệu NLTK (nếu chưa có)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


2. Đọc dữ liệu, kiểm tra cấu trúc, loại bỏ trùng lặp, phân tích độ dài

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
print("Kích thước:", df.shape)
print("\n5 dòng đầu:")
print(df.head())
print("\nPhân bố nhãn sentiment:\n", df['sentiment'].value_counts())

# Xóa dòng trùng lặp
initial_len = len(df)
df = df.drop_duplicates()
print(f"\nĐã xóa {initial_len - len(df)} dòng trùng. Kích thước mới: {df.shape}")

# Đếm số lượng mỗi nhãn sau khi xóa trùng
print("\nSố lượng nhãn sau khi xóa trùng:\n", df['sentiment'].value_counts())

# Tính độ dài (ký tự) và số từ
df['text_length'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

# Vẽ biểu đồ histogram phân phối độ dài
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['text_length'], bins=50, ax=axes[0], color='blue')
axes[0].set_title('Phân phối độ dài review (ký tự)')
sns.histplot(df['word_count'], bins=50, ax=axes[1], color='green')
axes[1].set_title('Phân phối độ dài review (từ)')
plt.tight_layout()
plt.show()
# Xác định những review quá ngắn (<10 từ)
short = df[df['word_count'] < 10]
print(f"\nSố lượng review có ít hơn 10 từ: {len(short)} (giữ lại)")
# (giữ lại vì có thể vẫn mang giá trị cảm xúc)

3. Viết hàm tiền xử lý văn bản

In [ ]:
# Khởi tạo tài nguyên tĩnh (chạy 1 lần)
STOP_WORDS = set(stopwords.words('english'))
NEGATION_WORDS = {'no', 'not', 'nor', 'never', 'without', "can't", "won't",
                  "shouldn't", "wouldn't", "couldn't", "doesn't", "didn't",
                  "isn't", "wasn't", "weren't", "hasn't", "haven't", "hadn't", "don't"}
STOP_WORDS = STOP_WORDS - NEGATION_WORDS

LEMMATIZER = WordNetLemmatizer()

def preprocess(text):
    # chuyển thành chữ thường
    text = text.lower()
    # loại bỏ thẻ HTML <br />
    text = re.sub(r'<br\s*/?>', ' ', text)
    # mở rộng từ viết tắt (contractions)
    text = contractions.fix(text)
    # loại bỏ số (thường không mang giá trị cảm xúc)
    text = re.sub(r'\d+', '', text)
    # loại bỏ dấu câu
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    # thu gọn nhiều khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    # tách từ (tokenize)
    tokens = text.split()
    # loại bỏ stopwords thông thường
    tokens = [t for t in tokens if t not in STOP_WORDS]
    # xử lý phủ định – tạo bigram (giữ nguyên logic cũ nhưng đã tối ưu)
    neg_handled = []
    i = 0
    while i < len(tokens):
        if tokens[i] in NEGATION_WORDS and i+1 < len(tokens):
            neg_handled.append(tokens[i] + '_' + tokens[i+1])
            i += 2
        else:
            neg_handled.append(tokens[i])
            i += 1
    # đưa về dạng gốc (lemmatization)
    lemmatized = [LEMMATIZER.lemmatize(tok) for tok in neg_handled]
    # ghép lại thành chuỗi
    return ' '.join(lemmatized)

# Thử nghiệm trên một review mẫu
sample = df.iloc[0]['review']
print("Gốc:", sample[:300], "...\n")
print("Đã xử lý:", preprocess(sample)[:300], "...")

4. Áp dụng tiền xử lý lên toàn bộ DataFrame

In [ ]:
print("Đang xử lý tất cả các bài đánh giá...")
df['cleaned_review'] = df['review'].apply(preprocess)
print("Hoàn tất!")

5. Phân tích khám phá dữ liệu (EDA)

In [ ]:
#phân phối độ dài theo nhãn
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[df['sentiment']=='positive']['word_count'],
             bins=60, color='steelblue', alpha=0.7, label='Positive')
axes[0].hist(df[df['sentiment']=='negative']['word_count'],
             bins=60, color='tomato',    alpha=0.7, label='Negative')
axes[0].axvline(df['word_count'].mean(), color='black',
                linestyle='--', label=f"Mean: {df['word_count'].mean():.0f}")
axes[0].set_xlabel('Số từ')
axes[0].set_ylabel('Số lượng review')
axes[0].set_title('Phân phối độ dài review theo nhãn')
axes[0].legend()

counts = df['sentiment'].value_counts()
axes[1].pie(counts, labels=counts.index,
            autopct='%1.1f%%', colors=['steelblue','tomato'],
            startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Tỷ lệ nhãn Positive / Negative')

plt.tight_layout()
plt.show()
# Vẽ top 20 từ chung
# Giả sử df['cleaned_review'] là cột đã tiền xử lý
all_words = ' '.join(df['cleaned_review']).split()
word_freq = Counter(all_words).most_common(20)

words, counts = zip(*word_freq) 
plt.figure(figsize=(10,6))
plt.bar(words, counts)
plt.title('Top 20 từ xuất hiện nhiều nhất toàn bộ dữ liệu')
plt.xticks(rotation=0,)
plt.show()

#  Top 20 từ phổ biến nhất theo nhãn ───────────────────────
def top_words(series, n=20):
    all_words = ' '.join(series).split()
    return pd.DataFrame(Counter(all_words).most_common(n),
                        columns=['word', 'count'])

top_pos = top_words(df[df['sentiment']=='positive']['cleaned_review'])
top_neg = top_words(df[df['sentiment']=='negative']['cleaned_review'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_pos['word'][::-1], top_pos['count'][::-1], color='steelblue')
axes[0].set_title('Top 20 từ — Positive')
axes[0].set_xlabel('Tần suất')

axes[1].barh(top_neg['word'][::-1], top_neg['count'][::-1], color='tomato')
axes[1].set_title('Top 20 từ — Negative')
axes[1].set_xlabel('Tần suất')

plt.tight_layout()
plt.show()


# WordCloud ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color in zip(axes,
                             ['positive', 'negative'],
                             ['Blues',    'Reds']):
    text = ' '.join(df[df['sentiment']==label]['review'])
    wc = WordCloud(width=700, height=400,
                   background_color='white',
                   colormap=color,
                   max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud — {label.capitalize()}', fontsize=13)

plt.tight_layout()
plt.show()

6. Tách đặc trưng và nhãn, chuyển nhãn thành số

In [ ]:
X = df['cleaned_review']
y = df['sentiment'].map({'positive': 1, 'negative': 0})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Lưu lại review gốc (chưa qua tiền xử lý) để dùng riêng cho BERT
X_raw = df['review']
# Dictionary lưu kết quả
cv_results = {}

7. Định nghĩa hàm đánh giá với cross‑validation và GridSearch

In [ ]:
def evaluate_model_with_cv(model, param_grid, model_name, X, y, skf):
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    # Lưu các giá trị qua các fold
    acc_folds, prec_folds, rec_folds, f1_folds = [], [], [], []
    best_estimators = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # TF-IDF
        X_train_tf = tfidf.fit_transform(X_train)
        X_val_tf = tfidf.transform(X_val)

        # GridSearch (chỉ thực hiện trên training fold)
        gs = GridSearchCV(model, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
        gs.fit(X_train_tf, y_train)
        best_model = gs.best_estimator_
        best_estimators.append(best_model)
        y_pred = best_model.predict(X_val_tf)

        acc = accuracy_score(y_val, y_pred)
        prec = precision_score(y_val, y_pred)
        rec = recall_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)

        acc_folds.append(acc)
        prec_folds.append(prec)
        rec_folds.append(rec)
        f1_folds.append(f1)

        print(f"{model_name} Lần thứ {fold+1}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

    print(f"\n{model_name} – Trung bình: Acc={np.mean(acc_folds):.4f}±{np.std(acc_folds):.4f}, "
          f"Prec={np.mean(prec_folds):.4f}, Rec={np.mean(rec_folds):.4f}, F1={np.mean(f1_folds):.4f}")

    return {
        'accuracy': (np.mean(acc_folds), np.std(acc_folds)),
        'precision': np.mean(prec_folds),
        'recall': np.mean(rec_folds),
        'f1': np.mean(f1_folds),
        'all_folds': acc_folds
    }

8. Huấn luyện mô hình Naive Bayes

In [ ]:

nb_param = {'alpha': [0.1, 0.5, 1.0]}
nb = MultinomialNB()
cv_results['Naive Bayes'] = evaluate_model_with_cv(nb, nb_param, 'Naive Bayes', X, y, skf)

9. Huấn luyện mô hình Logistic Regression

In [ ]:
lr_param = {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear']}
lr = LogisticRegression(max_iter=1000)
cv_results['Logistic Regression'] = evaluate_model_with_cv(lr, lr_param, 'Logistic Regression', X, y, skf)

10. Huấn luyện mô hình Random Forest

In [ ]:
rf_param = {'n_estimators': [50], 'max_depth': [10]}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

cv_results['Random Forest'] = evaluate_model_with_cv(rf, rf_param, 'Random Forest', X, y, skf)

11. Tạo Dataset cho DistilBERT

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts          # texts là review GỐC (chưa xử lý)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        # Chỉ làm sạch tối thiểu: bỏ HTML, chuẩn hóa whitespace
        text = re.sub(r'<br\s*/?>', ' ', text)
        text = contractions.fix(text)
        text = re.sub(r'\s+', ' ', text).strip()
        # Tokenize (giữ nguyên chữ hoa/thường, dấu câu)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

12. Tinh chỉnh DistilBERT

In [ ]:

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print("Đang chuẩn bị dữ liệu cho DistilBERT...")

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

train_dataset = IMDBDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = IMDBDataset(X_test.tolist(), y_test.tolist(), tokenizer)

print(f"Tập huấn luyện: {len(train_dataset)} mẫu")
print(f"Tập kiểm tra: {len(test_dataset)} mẫu")

# Cấu hình training
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available()
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# Tạo Trainer (bỏ tokenizer)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

print("Bắt đầu huấn luyện DistilBERT...")
trainer.train()

print("Đánh giá mô hình...")
bert_metrics = trainer.evaluate()
print(f"Độ chính xác của DistilBERT: {bert_metrics['eval_accuracy']:.4f} ({bert_metrics['eval_accuracy']*100:.2f}%)")

13. So sánh kết quả các mô hình

In [ ]:
bert_acc = bert_metrics['eval_accuracy']
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
# Tính các chỉ số
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
# Tạo bảng so sánh
comparison = pd.DataFrame({
    'Mô hình': list(cv_results.keys()) + ['DistilBERT'],
    'Độ chính xác (trung bình±độ lệch)': [f"{v['accuracy'][0]:.4f}±{v['accuracy'][1]:.4f}" for v in cv_results.values()] + [f"{bert_acc:.4f}±N/A"],
    'Precision': [f"{v['precision']:.4f}" for v in cv_results.values()] + [f"{precision:.4f}"],
    'Recall': [f"{v['recall']:.4f}" for v in cv_results.values()] + [f"{recall:.4f}"],
    'F1-score': [f"{v['f1']:.4f}" for v in cv_results.values()] + [f"{f1:.4f}"]
})
print(comparison.to_string(index=False))

14. Phân tích lỗi trên Logistic Regression

In [ ]:

# Huấn luyện lại Logistic Regression trên toàn bộ tập train (80%) và đánh giá trên tập test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

best_lr = LogisticRegression(C=1, solver='liblinear', max_iter=1000)
best_lr.fit(X_train_tf, y_train)
y_pred = best_lr.predict(X_test_tf)
y_proba = best_lr.predict_proba(X_test_tf)[:, 1]

# Tìm các mẫu bị phân loại sai
misclassified = (y_pred != y_test)
mis_idx = np.where(misclassified)[0]

print(f"Tổng số mẫu sai: {len(mis_idx)} / {len(y_test)} ({len(mis_idx)/len(y_test)*100:.2f}%)")

# Lấy 15 mẫu sai đầu tiên
sample_errors = []
for i in mis_idx[:15]:
    sample_errors.append({
        'review_gốc': X_test.iloc[i][:200] + "...",
        'nhãn_thực': 'positive' if y_test.iloc[i]==1 else 'negative',
        'nhãn_dự_đoán': 'positive' if y_pred[i]==1 else 'negative',
        'xác_suất': y_proba[i] if y_pred[i]==1 else 1-y_proba[i],
        'độ_tin_cậy': 'cao' if (y_proba[i] > 0.9 or y_proba[i] < 0.1) else 'trung bình'
    })

# Hiển thị bảng các mẫu sai
pd.DataFrame(sample_errors)

15. Gắn nhãn thực thể (NER) trên 5000 review mẫu

In [ ]:

sample_ner = df.sample(5000, random_state=42)
entity_counts = {'PERSON': 0, 'ORG': 0, 'GPE': 0, 'DATE': 0, 'PRODUCT': 0}
pos_entities = {'PERSON': [], 'ORG': [], 'GPE': []}
neg_entities = {'PERSON': [], 'ORG': [], 'GPE': []}

for idx, row in sample_ner.iterrows():
    doc = nlp(row['review'][:1000])  # giới hạn độ dài để tiết kiệm thời gian
    for ent in doc.ents:
        if ent.label_ in entity_counts:
            entity_counts[ent.label_] += 1
            if row['sentiment'] == 'positive' and ent.label_ in pos_entities:
                pos_entities[ent.label_].append(ent.text)
            elif row['sentiment'] == 'negative' and ent.label_ in neg_entities:
                neg_entities[ent.label_].append(ent.text)

print("Tần suất xuất hiện của các thực thể:", entity_counts)
print("\n10 thực thể PERSON xuất hiện nhiều nhất trong review tích cực:", Counter(pos_entities['PERSON']).most_common(10))
print("\n10 thực thể PERSON xuất hiện nhiều nhất trong review tiêu cực:", Counter(neg_entities['PERSON']).most_common(10))

16. Vẽ các biểu đồ chính

In [ ]:
# Biểu đồ : So sánh độ chính xác các mô hình (5-fold CV)
models = list(cv_results.keys())
acc_means = [cv_results[m]['accuracy'][0] for m in models]
acc_stds = [cv_results[m]['accuracy'][1] for m in models]
plt.figure(figsize=(8,5))
plt.bar(models, acc_means, yerr=acc_stds, capsize=5)
plt.ylabel('Độ chính xác (Accuracy)')
plt.title('So sánh các mô hình với 5-fold cross-validation')
plt.ylim(0.8, 0.95)
plt.show()

# Biểu đồ : Ma trận nhầm lẫn cho Logistic Regression
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn - Logistic Regression')
plt.show()

# Biểu đồ : Đặc trưng quan trọng nhất (hệ số hồi quy)
feature_names = tfidf.get_feature_names_out()
coefs = best_lr.coef_.flatten()
pos_coef = sorted(zip(feature_names, coefs), key=lambda x: x[1], reverse=True)[:20]
neg_coef = sorted(zip(feature_names, coefs), key=lambda x: x[1])[:20]

fig, ax = plt.subplots(1,2, figsize=(12,6))
ax[0].barh([p[0] for p in pos_coef[::-1]], [p[1] for p in pos_coef[::-1]], color='green')
ax[0].set_title('20 đặc trưng tích cực nhất')
ax[1].barh([n[0] for n in neg_coef[::-1]], [n[1] for n in neg_coef[::-1]], color='red')
ax[1].set_title('20 đặc trưng tiêu cực nhất')
plt.tight_layout()
plt.show()

# Biểu đồ : Đường cong ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('Tỷ lệ tích cực giả (FPR)')
plt.ylabel('Tỷ lệ tích cực thật (TPR)')
plt.title('Đường cong ROC')
plt.legend()
plt.show()

17. Xây dựng hàm dự đoán cảm xúc

In [ ]:

def predict_sentiment(text):
    # Sử dụng đúng pipeline tiền xử lý
    cleaned = preprocess(text)
    # Chuyển thành vector bằng TF-IDF đã fit trên toàn bộ tập train (X_train_tf)
    # Lưu ý: cần vectorizer đã fit từ cell 13 – nếu chưa có thì fit lại
    if 'tfidf' not in dir():
        global tfidf
        tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
        tfidf.fit(X_train)
    vec = tfidf.transform([cleaned])
    proba = best_lr.predict_proba(vec)[0][1]
    pred = 1 if proba >= 0.5 else 0
    return ("Tích cực" if pred == 1 else "Tiêu cực"), proba if pred==1 else 1-proba

# Thử nghiệm với các câu mẫu
test_phrases = [
    "I absolutely loved this movie, it was fantastic!",
    "Worst film ever, waste of time.",
    "Not bad at all, actually pretty good.",
    "I can't believe how terrible the acting was. Avoid at all costs.",
    "It was okay, nothing special."
]
for phrase in test_phrases:
    sent, conf = predict_sentiment(phrase)
    print(f"'{phrase}' -> {sent} (độ tin cậy: {conf:.3f})")